# High-Level Distributions

This notebook demonstrates composite distributions built from basic ones:

- `Independent` - product of independent distributions
- `Mixture` - mixture (weighted combination) of distributions
- `Transformed` - apply bijective transformations

These allow building complex distributions from simple components.

In [ ]:
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt

import probjax.stats as stats
from probjax.stats.independent import Independent
from probjax.stats.mixture import Mixture
from probjax.stats.transformed import Transformed
from probjax.stats.bijective import affine

key = jax.random.PRNGKey(0)

## Independent Distribution

Create a product distribution where each dimension is independent.

In [ ]:
# Independent Normals with different parameters
components = [
    stats.norm(loc=-2., scale=0.5),
    stats.norm(loc=0., scale=1.),
    stats.norm(loc=3., scale=0.8)
]

p = Independent(components)

samples = p.rvs(key, shape=(5000,))

fig, axes = plt.subplots(1, 3, figsize=(12, 3))
for i, ax in enumerate(axes):
    ax.hist(samples[:, i], bins=30, density=True, alpha=0.7)
    ax.set_title(f'Dimension {i+1}')
    ax.set_xlabel('x')
plt.tight_layout()
plt.show()

print(f"Batch shape: {p.batch_shape}")
print(f"Event shape: {p.event_shape}")
print(f"Mean: {p.mean}")
print(f"Var: {p.var}")

In [ ]:
# Can also create from a single distribution with batch shape
base_dist = stats.norm(loc=jnp.array([-2., 0., 3.]), scale=jnp.array([0.5, 1., 0.8]))
p2 = Independent([base_dist])

samples2 = p2.rvs(key, shape=(5000,))
print(f"Samples shape: {samples2.shape}")
print(f"Same distribution: {jnp.allclose(samples.mean(0), samples2.mean(0), atol=0.1)}")

## Mixture Distribution

Create a mixture (weighted combination) of multiple distributions.

In [ ]:
# Gaussian Mixture Model (GMM)
components = [
    stats.norm(loc=-3., scale=0.5),
    stats.norm(loc=0., scale=1.0),
    stats.norm(loc=4., scale=0.7)
]
weights = jnp.array([0.3, 0.5, 0.2])

p = Mixture(components, weights=weights)

samples = p.rvs(key, shape=(10000,))

plt.hist(samples, bins=50, density=True, alpha=0.7, label='Mixture samples')

# Plot individual components
xs = jnp.linspace(-6, 7, 200)
for i, (comp, w) in enumerate(zip(components, weights)):
    plt.plot(xs, w * jnp.exp(comp.logpdf(xs)), '--', label=f'Component {i+1} (w={w:.1f})')

plt.plot(xs, jnp.exp(p.logpdf(xs)), 'r-', linewidth=2, label='Mixture PDF')
plt.xlabel('x')
plt.ylabel('Density')
plt.title('Gaussian Mixture Model')
plt.legend()
plt.show()

print(f"Mean: {p.mean:.3f}")
print(f"Var: {p.var:.3f}")

In [ ]:
# 2D Mixture of Multivariate Normals
components_2d = [
    stats.multivariate_normal(
        mean=jnp.array([-2., -2.]),
        cov=jnp.array([[0.5, 0.], [0., 0.5]])
    ),
    stats.multivariate_normal(
        mean=jnp.array([2., 2.]),
        cov=jnp.array([[0.8, 0.3], [0.3, 0.8]])
    ),
    stats.multivariate_normal(
        mean=jnp.array([0., 3.]),
        cov=jnp.array([[0.3, 0.], [0., 0.3]])
    )
]
weights_2d = jnp.array([0.4, 0.4, 0.2])

p2d = Mixture(components_2d, weights=weights_2d)

samples_2d = p2d.rvs(key, shape=(5000,))

plt.figure(figsize=(8, 6))
plt.scatter(samples_2d[:, 0], samples_2d[:, 1], alpha=0.3, s=5)
plt.xlabel('x1')
plt.ylabel('x2')
plt.title('2D Gaussian Mixture')
plt.axis('equal')
plt.show()

print(f"Mean: {p2d.mean}")
print(f"Cov shape: {p2d.cov.shape}")

## Transformed Distribution

Apply bijective transformations to a base distribution.

In [ ]:
# Log-normal distribution via exp transform
base = stats.norm(loc=0., scale=1.)

# Create transformed distribution with exp bijector
exp_transform = lambda x: jnp.exp(x)
log_transform = lambda y: jnp.log(y)

p = Transformed(
    base,
    forward_fn=exp_transform,
    inverse_fn=log_transform,
    log_det_jacobian_fn=lambda x: -jnp.log(x)  # derivative of log is 1/x
)

samples = p.rvs(key, shape=(10000,))

plt.hist(samples, bins=50, density=True, alpha=0.7, label='Samples')

xs = jnp.linspace(0.1, 5, 100)
plt.plot(xs, jnp.exp(p.logpdf(xs)), 'r-', label='LogNormal PDF')
plt.xlabel('x')
plt.ylabel('Density')
plt.title('Log-Normal (exp of Normal)')
plt.legend()
plt.show()

print(f"Mean: {p.mean:.3f}")
print(f"Var: {p.var:.3f}")

In [ ]:
# Affine transformation: Shift and scale
base = stats.norm(loc=0., scale=1.)

# Transform: y = 2*x + 5
shift = 5.0
scale = 2.0

p = Transformed(
    base,
    forward_fn=lambda x: scale * x + shift,
    inverse_fn=lambda y: (y - shift) / scale,
    log_det_jacobian_fn=lambda x: jnp.log(jnp.abs(scale))
)

samples = p.rvs(key, shape=(10000,))

plt.hist(samples, bins=50, density=True, alpha=0.7, label='Transformed samples')

xs = jnp.linspace(-2, 12, 100)
plt.plot(xs, jnp.exp(p.logpdf(xs)), 'r-', label='PDF')
plt.xlabel('x')
plt.ylabel('Density')
plt.title('Affine Transform: Normal → Normal(5, 2)')
plt.legend()
plt.show()

print(f"Mean: {p.mean:.3f} (expected: 5.0)")
print(f"Std: {p.std:.3f} (expected: 2.0)")

In [ ]:
# Sigmoid transform to create a bounded distribution
base = stats.norm(loc=0., scale=2.)

# Transform through sigmoid: maps R → (0, 1)
p = Transformed(
    base,
    forward_fn=jax.nn.sigmoid,
    inverse_fn=lambda y: jnp.log(y / (1 - y)),
    log_det_jacobian_fn=lambda x: -jax.nn.softplus(x) - jax.nn.softplus(-x)
)

samples = p.rvs(key, shape=(10000,))

plt.hist(samples, bins=50, density=True, alpha=0.7, label='Samples')

xs = jnp.linspace(0.01, 0.99, 100)
plt.plot(xs, jnp.exp(p.logpdf(xs)), 'r-', label='Sigmoid-Normal PDF')
plt.xlabel('x')
plt.ylabel('Density')
plt.title('Sigmoid of Normal(0, 2)')
plt.legend()
plt.show()

print(f"Samples in (0, 1): {((samples > 0) & (samples < 1)).all()}")
print(f"Mean: {p.mean:.3f}")

## Composition: Complex Distributions

Combine Independent, Mixture, and Transformed for more complex models.

In [ ]:
# Mixture of transformed distributions
# Component 1: Log-normal (exp of Normal)
comp1 = Transformed(
    stats.norm(loc=0., scale=0.5),
    forward_fn=jnp.exp,
    inverse_fn=jnp.log,
    log_det_jacobian_fn=lambda x: -jnp.log(x)
)

# Component 2: Shifted exponential
comp2 = Transformed(
    stats.expon(scale=1.),
    forward_fn=lambda x: x + 2,
    inverse_fn=lambda y: y - 2,
    log_det_jacobian_fn=lambda x: 0.
)

# Component 3: Scaled beta
comp3 = Transformed(
    stats.beta(a=2., b=2.),
    forward_fn=lambda x: 5 * x,
    inverse_fn=lambda y: y / 5,
    log_det_jacobian_fn=lambda x: jnp.log(5.)
)

p_complex = Mixture([comp1, comp2, comp3], weights=jnp.array([0.3, 0.4, 0.3]))

samples = p_complex.rvs(key, shape=(10000,))

plt.hist(samples, bins=50, density=True, alpha=0.7, label='Mixture samples')
plt.xlabel('x')
plt.ylabel('Density')
plt.title('Mixture of Transformed Distributions')
plt.legend()
plt.show()

print(f"Mean: {p_complex.mean:.3f}")
print(f"Var: {p_complex.var:.3f}")

## Summary

High-level distributions compose basic ones:

| Class | Purpose | Example |
|-------|---------|---------|
| `Independent` | Product distribution | IID normals |
| `Mixture` | Weighted combination | GMM |
| `Transformed` | Bijective transform | Log-normal |

These enable:
- **Modular modeling**: Build complex distributions from simple components
- **Automatic inference**: Transformations preserve probability (change of variables)
- **Flexible sampling**: All support `rvs()` with the same interface

For variational inference and probabilistic modeling, these primitives are essential building blocks.